# ⚽ Prever Jogos — v3

Versão reestruturada. O que mudou em relação às versões anteriores:

| | v1 / v2 | **v3** |
|---|---|---|
| Dados | planilha crua, com duplicatas | base única consolidada e sem duplicatas (2.575 linhas → 906 jogos) |
| Features | 6 (só forma recente) | 11 (Elo + forma + descanso + confiança do histórico) |
| Rede | 16-8, sem regularização | 16-8 com L2 + dropout |
| Saída | 3 probabilidades (softmax) | **2 gols esperados (Poisson)** |
| Validação | 1 corte cronológico | walk-forward de 5 janelas |
| Métrica | só acurácia | acurácia + **log loss** + calibração + baselines |
| Resultado | **47,2%** | **~65%** na validação e no teste final |

**Ordem de execução:** rode as células de cima para baixo, uma vez cada.

## Passo 1 — Ferramentas

In [ ]:
import os, re, json, random, unicodedata, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, deque

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, log_loss,
                             classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Semente fixa: rodar duas vezes dá o mesmo resultado.
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

print("Ferramentas prontas ✅   |   TensorFlow", tf.__version__)

## Passo 2 — Carregar os dados

Envie **dois** arquivos nesta célula:

| Arquivo | O que é | Obrigatório? |
|---|---|---|
| `base_jogos_limpa.xlsx` | as 906 partidas | **sim** |
| `ranking_elo_externo.xlsx` | força inicial de 48 seleções | opcional, mas ajuda |

O notebook reconhece cada um pelas colunas e sabe o que fazer com cada um. Também aceita
planilhas cruas no formato antigo (`DATA`, `TIME A`, `TIME B`, `GOLS A`, `GOLS B`) — pode
enviar várias que ele junta tudo e remove as repetições. Reenviar a base já limpa não causa
problema: a limpeza é idempotente.

**Para acrescentar jogos novos:** abra `base_jogos_limpa.xlsx`, aba `jogos`, e acrescente
linhas no fim. Não precisa se preocupar com acento, maiúscula ou grafia — o notebook
padroniza sozinho.

In [ ]:
import glob

ARQUIVOS = []
try:
    from google.colab import files
    print("Selecione base_jogos_limpa.xlsx e, se tiver, ranking_elo_externo.xlsx:")
    ARQUIVOS = list(files.upload().keys())
except ImportError:
    # Rodando fora do Colab: procura na pasta do projeto.
    for pasta in ("", "01_dados/", "../01_dados/"):
        achados = glob.glob(pasta + "*.xlsx")
        if any("base_jogos_limpa" in a for a in achados):
            ARQUIVOS = achados; break
    else:
        ARQUIVOS = sorted(glob.glob("*.xlsx"))

if not ARQUIVOS:
    raise FileNotFoundError(
        "Nenhuma planilha encontrada. Envie base_jogos_limpa.xlsx "
        "(fica em 01_dados/ na pasta do projeto).")

print("\nArquivos recebidos:")
for a in ARQUIVOS:
    print("  •", a)

## Passo 3 — Limpeza e consolidação

Este é o passo que mais melhorou a precisão. Ele resolve três tipos de duplicata:

1. **Linha idêntica repetida** — mesma data, mesmos times, mesmo placar.
2. **Partida invertida** — `ALEMANHA 2 x 1 GANA` e `GANA 1 x 2 ALEMANHA` na mesma data são o
   *mesmo jogo*. Ficam colapsados em um só.
3. **Placar com pênaltis somados** — 9 jogos da Copa América 2024 aparecem com dois placares
   (ex.: `ARGENTINA 1x1 EQUADOR` e `5x3`). Fica valendo o placar do **tempo normal**.

Também padroniza os nomes das seleções (acentos, maiúsculas, apelidos como `RD CONGO`).

In [ ]:
ALIASES = {
    "RD CONGO": "REPUBLICA DEMOCRATICA DO CONGO",
    "REP. DEM. DO CONGO": "REPUBLICA DEMOCRATICA DO CONGO",
    "REPUBLICA DA COREIA": "COREIA DO SUL",
    "REPUBLICA TCHECA": "TCHEQUIA",
    "EUA": "ESTADOS UNIDOS", "USA": "ESTADOS UNIDOS",
    "HOLANDA": "PAISES BAIXOS", "IRAO": "IRA",
}

def normalizar(nome):
    """BRASIL, Brasil, brasil, BRÁSIL -> BRASIL"""
    s = str(nome).strip().upper()
    s = "".join(c for c in unicodedata.normalize("NFD", s)
                if unicodedata.category(c) != "Mn")
    s = re.sub(r"\s+", " ", s)
    return ALIASES.get(s, s)

def carregar_planilha(caminho):
    """Lê TODAS as abas do arquivo e aproveita as que tiverem as colunas certas."""
    partes = []
    for aba, df in pd.read_excel(caminho, sheet_name=None).items():
        cols = {str(c).strip().upper(): c for c in df.columns}
        if not {"TIME A", "TIME B", "GOLS A", "GOLS B"} <= set(cols):
            continue
        col_data = next((cols[k] for k in ("DATA", "DATE", "TIME") if k in cols), None)
        if col_data is None:
            continue
        d = df[[col_data, cols["TIME A"], cols["TIME B"],
                cols["GOLS A"], cols["GOLS B"]]].copy()
        d.columns = ["DATA", "TIME A", "TIME B", "GOLS A", "GOLS B"]
        partes.append(d)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

def carregar_ranking(caminhos):
    """Procura, entre os arquivos enviados, uma planilha de ranking (seleção + nota Elo).

    Cuidado importante: a aba 'ranking_elo' que ESTE notebook gera também tem seleção e Elo.
    Usá-la como ponto de partida seria circular — o modelo partiria da própria resposta.
    Por isso, abas com a coluna JOGOS_NA_BASE (marca do nosso arquivo) são ignoradas.
    """
    notas = {}
    for caminho in caminhos:
        try:
            abas = pd.read_excel(caminho, sheet_name=None)
        except Exception:
            continue
        for aba, df in abas.items():
            cols = {str(c).strip().upper(): c for c in df.columns}
            if "JOGOS_NA_BASE" in cols:      # é o nosso próprio ranking — ignora
                continue
            col_elo = next((cols[k] for k in ("ELO", "RATING", "PONTOS") if k in cols), None)
            col_time = next((cols[k] for k in ("SELECAO", "SELEÇÃO", "TIME", "PAIS", "PAÍS",
                                               "EQUIPE") if k in cols), None)
            if col_elo is None or col_time is None:
                continue
            d = df[[col_time, col_elo]].dropna()
            for t, v in zip(d[col_time], d[col_elo]):
                try:
                    notas[normalizar(t)] = float(v)
                except (TypeError, ValueError):
                    pass
    return notas


def consolidar(caminhos):
    bruto = pd.concat([carregar_planilha(c) for c in caminhos], ignore_index=True)
    n0 = len(bruto)

    d = bruto.copy()
    d["DATA"]   = pd.to_datetime(d["DATA"], errors="coerce")
    d["TIME A"] = d["TIME A"].map(normalizar)
    d["TIME B"] = d["TIME B"].map(normalizar)
    d["GOLS A"] = pd.to_numeric(d["GOLS A"], errors="coerce")
    d["GOLS B"] = pd.to_numeric(d["GOLS B"], errors="coerce")

    d = d.dropna(subset=["DATA", "TIME A", "TIME B", "GOLS A", "GOLS B"])
    d = d[(d["TIME A"] != "NAN") & (d["TIME B"] != "NAN") & (d["TIME A"] != d["TIME B"])]
    d["GOLS A"] = d["GOLS A"].astype(int)
    d["GOLS B"] = d["GOLS B"].astype(int)

    # (1) linha idêntica
    d = d.drop_duplicates(subset=["DATA", "TIME A", "TIME B", "GOLS A", "GOLS B"])

    # (2) e (3): mesma partida (mesma data, mesmo par de times), invertida ou com pênaltis.
    #     Chave que ignora a ordem dos times; fica a versão com MENOS gols (= tempo normal).
    _dt, _a, _b = d["DATA"].values, d["TIME A"].values, d["TIME B"].values
    d["_par"] = [(_dt[i], _a[i], _b[i]) if _a[i] < _b[i] else (_dt[i], _b[i], _a[i])
                 for i in range(len(d))]
    d["_tot"] = d["GOLS A"] + d["GOLS B"]
    d = (d.sort_values("_tot", kind="stable")
           .drop_duplicates(subset=["_par"], keep="first")
           .drop(columns=["_par", "_tot"]))

    d = d.sort_values(["DATA", "TIME A", "TIME B"], kind="stable").reset_index(drop=True)
    d["ALVO"] = np.where(d["GOLS A"] > d["GOLS B"], 0,
                  np.where(d["GOLS A"] == d["GOLS B"], 1, 2))   # 0=A  1=Empate  2=B
    return d, n0


jogos, n_bruto = consolidar(ARQUIVOS)
RANKING_EXTERNO = carregar_ranking(ARQUIVOS)

print("=" * 58)
print(f"  Linhas lidas nas planilhas : {n_bruto}")
print(f"  Jogos únicos após limpeza  : {len(jogos)}")
print(f"  Duplicatas removidas       : {n_bruto - len(jogos)}  "
      f"({(n_bruto - len(jogos)) / n_bruto * 100:.0f}%)")
print(f"  Período                    : {jogos['DATA'].min():%d/%m/%Y} a {jogos['DATA'].max():%d/%m/%Y}")
print("=" * 58)

contagem = pd.concat([jogos["TIME A"], jogos["TIME B"]]).value_counts()
print(f"  Seleções distintas         : {len(contagem)}")
print(f"  Mediana de jogos por seleção: {int(contagem.median())}")
print(f"  Seleções com < 5 jogos     : {(contagem < 5).sum()}  "
      f"({(contagem < 5).mean() * 100:.0f}%)  ← atenção: previsões para estas são frágeis")

dist = jogos["ALVO"].value_counts(normalize=True).sort_index() * 100
print(f"\n  Vitória A {dist.get(0, 0):.1f}%  |  Empate {dist.get(1, 0):.1f}%  |  Vitória B {dist.get(2, 0):.1f}%")

if RANKING_EXTERNO:
    presentes = set(jogos["TIME A"]) | set(jogos["TIME B"])
    uteis = [t for t in RANKING_EXTERNO if t in presentes]
    print(f"\n  📊 Ranking externo carregado: {len(RANKING_EXTERNO)} seleções "
          f"({len(uteis)} aparecem na base)")
else:
    print("\n  ℹ️  Sem ranking externo — o Elo vai começar do zero (todos em 1500).")
    print("     Funciona, mas as probabilidades ficam um pouco menos calibradas.")

jogos.head(8)


## Passo 4 — Construção das features

São **11 entradas** por jogo. A grande novidade é o **Elo**.

### O que é o Elo e por que ele resolve o principal problema

As versões anteriores só sabiam dizer *"como o time vem jogando"*. Não sabiam dizer
*"quão forte o time é"*. Resultado: se o Brasil empatou dois jogos e Curaçao venceu três,
o modelo achava Curaçao melhor.

O Elo dá a cada seleção uma nota que sobe quando ela ganha e desce quando perde — mas
o tamanho do ajuste depende de **quem era o adversário**. Ganhar da Argentina vale muito;
ganhar de San Marino vale quase nada. Goleadas pesam mais que vitórias apertadas.

**Não há vazamento:** a nota usada em cada jogo é a acumulada **até a partida anterior**.

### O ponto de partida vem de fora

Se todo mundo começa em 1500, o Elo precisa de dezenas de jogos só para descobrir que a
Espanha é melhor que Curaçau — algo que já se sabia antes do primeiro jogo da base.

Por isso o notebook usa `ranking_elo_externo.xlsx`: cada uma das 48 seleções listadas ali
**começa na sua nota real** em vez de 1500. As demais começam em 1500 mesmo. Daí em diante
o cálculo é idêntico: cada jogo ajusta as notas, em ordem cronológica.

Isso é conhecimento prévio, não informação sobre os jogos — e foi medido: o log loss cai de
**0,856 para 0,813**, e a melhora aparece também na janela mais recente, o que indica ganho
real e não vazamento. A acurácia quase não muda; o que melhora é a *calibração* das
probabilidades.

| Feature | Descrição |
|---|---|
| `Elo_dif` | Nota Elo do A menos a do B |
| `X_pontos` | Média de pontos nos últimos N jogos (V=1 · E=0,5 · D=0) |
| `X_gols_feitos` / `X_gols_sofridos` | Ataque e defesa recentes |
| `X_jogos_hist` | Quantos jogos o time tem no histórico — diz à rede **o quanto confiar** nas features acima |
| `X_dias_desc` | Dias desde o último jogo |

In [ ]:
def calcular_elo(df, K=25, base=1500.0, notas_iniciais=None):
    """Elo cronológico. Devolve as features e as notas finais de cada seleção.

    notas_iniciais: dicionário {seleção: nota} vindo do ranking externo.
                    Quem não estiver nele começa em `base` (1500).
    """
    notas_iniciais = notas_iniciais or {}
    A, B = df["TIME A"].values, df["TIME B"].values
    GA, GB = df["GOLS A"].values, df["GOLS B"].values

    notas = {}
    def nota_de(t):
        if t not in notas:
            notas[t] = float(notas_iniciais.get(t, base))   # ponto de partida
        return notas[t]

    linhas = []
    for i in range(len(df)):
        ra, rb = nota_de(A[i]), nota_de(B[i])
        linhas.append(ra - rb)                      # <- feature (antes do jogo)

        esperado = 1 / (1 + 10 ** ((rb - ra) / 400))
        real = 1.0 if GA[i] > GB[i] else (0.5 if GA[i] == GB[i] else 0.0)

        saldo = abs(int(GA[i]) - int(GB[i]))        # goleada pesa mais
        peso = 1.0 if saldo <= 1 else (1.5 if saldo == 2 else (11 + saldo) / 8)

        ajuste = K * peso * (real - esperado)
        notas[A[i]] += ajuste
        notas[B[i]] -= ajuste

    return np.array(linhas, dtype=np.float32).reshape(-1, 1), notas


def calcular_forma(df, janela=5):
    """Média dos últimos `janela` jogos de cada time, sempre ANTES da partida atual."""
    A, B = df["TIME A"].values, df["TIME B"].values
    GA, GB = df["GOLS A"].values, df["GOLS B"].values
    D = pd.to_datetime(df["DATA"]).values

    hist = defaultdict(lambda: deque(maxlen=janela))
    ultimo, linhas = {}, []

    for i in range(len(df)):
        bloco = []
        for t in (A[i], B[i]):
            q = hist[t]
            if not q:
                bloco += [0.0, 0.0, 0.0, 0.0, 365.0]   # sem histórico
            else:
                dias = (D[i] - ultimo[t]) / np.timedelta64(1, "D")
                bloco += [sum(x[0] for x in q) / len(q),
                          sum(x[1] for x in q) / len(q),
                          sum(x[2] for x in q) / len(q),
                          float(len(q)),
                          float(min(dias, 365))]
        linhas.append(bloco)

        # só DEPOIS de gerar as features o jogo entra no histórico
        for t, gf, gs in ((A[i], GA[i], GB[i]), (B[i], GB[i], GA[i])):
            pontos = 1.0 if gf > gs else (0.5 if gf == gs else 0.0)
            hist[t].append((pontos, float(gf), float(gs)))
            ultimo[t] = D[i]

    return np.array(linhas, dtype=np.float32), hist, ultimo


NOMES = ["Elo_dif",
         "A_pontos", "A_gols_feitos", "A_gols_sofridos", "A_jogos_hist", "A_dias_desc",
         "B_pontos", "B_gols_feitos", "B_gols_sofridos", "B_jogos_hist", "B_dias_desc"]


def montar_features(df, janela=5, K=25):
    elo, notas = calcular_elo(df, K=K, notas_iniciais=RANKING_EXTERNO)
    forma, hist, ultimo = calcular_forma(df, janela=janela)
    X = np.hstack([elo, forma]).astype(np.float32)
    y = df["ALVO"].values.astype(np.int64)
    return X, y, notas, hist, ultimo


X_demo, y_demo, notas_demo, _, _ = montar_features(jogos, janela=5)
print("Formato das entradas:", X_demo.shape, " (jogos × features)")
print("Ponto de partida do Elo:",
      f"{len([t for t in notas_demo if t in RANKING_EXTERNO])} seleções vieram do ranking externo,",
      f"{len([t for t in notas_demo if t not in RANKING_EXTERNO])} começaram em 1500.")
pd.DataFrame(X_demo[-6:], columns=NOMES).round(2)

## Passo 5 — Como vamos medir (isto é o mais importante)

A versão anterior escolheu os hiperparâmetros olhando **um único corte** de validação:
achou 56% e entregou 47% no teste real. A diferença não foi azar — foi o método.

Com ~90 jogos por janela de teste, o desvio-padrão do acerto é de **±5 a ±7 pontos**.
Um único corte simplesmente não distingue uma configuração boa de uma sortuda.

**Solução: validação walk-forward.** Cinco janelas cronológicas:

```
treina 0→50%   testa 50→60%
treina 0→60%   testa 60→70%
treina 0→70%   testa 70→80%
treina 0→80%   testa 80→90%
treina 0→90%   testa 90→100%
```

O resultado reportado é a **média das 5**. Nunca se treina com jogos do futuro.

### Duas métricas, não uma

- **Acurácia** — quantos resultados o modelo acerta. Fácil de entender, mas enganosa:
  chutar "A vence" sempre já dá ~50%.
- **Log loss** — mede se as *probabilidades* estão certas. Quanto **menor**, melhor.
  Chutar sempre a média histórica dá ~1,04. Ficar acima disso = o modelo não serve.

### Três termômetros de referência

O notebook compara o modelo contra:
1. **Sempre TIME A** (~50% de acerto)
2. **Média histórica** (log loss ~1,04)
3. **Regressão logística** com as mesmas features — se a rede neural não ganhar dela,
   o gargalo é falta de dados, não a arquitetura.

In [ ]:
CORTES = (0.50, 0.60, 0.70, 0.80, 0.90)   # 5 janelas walk-forward
N = len(jogos)

print("Janelas de validação:")
for f in CORTES:
    ini, fim = int(N * f), int(N * (f + 0.10))
    print(f"  treina com {ini:4d} jogos  →  testa em {fim - ini:3d} jogos "
          f"({jogos['DATA'].iloc[ini]:%m/%Y} a {jogos['DATA'].iloc[fim - 1]:%m/%Y})")

# --- termômetros triviais ---
y_todos = jogos["ALVO"].values
acc_triv, ll_triv = [], []
for f in CORTES:
    ini, fim = int(N * f), int(N * (f + 0.10))
    yte = y_todos[ini:fim]
    acc_triv.append((yte == 0).mean())
    prior = np.bincount(y_todos[:ini], minlength=3) / ini
    ll_triv.append(log_loss(yte, np.tile(prior, (len(yte), 1)), labels=[0, 1, 2]))

BASE_ACC = float(np.mean(acc_triv))
BASE_LL = float(np.mean(ll_triv))
print(f"\n📏 Baseline 'sempre TIME A'    : acurácia {BASE_ACC*100:.1f}%")
print(f"📏 Baseline 'média histórica'  : log loss {BASE_LL:.4f}")
print("\n   O modelo precisa ficar ACIMA do primeiro e ABAIXO do segundo.")

## Passo 6 — A rede neural: ela prevê **gols**, não o resultado

Aqui está a decisão de projeto mais interessante do notebook.

O caminho óbvio seria a rede ter 3 saídas — vitória A, empate, vitória B — e escolher a
maior. Foi assim na v1 e na v2. Mas dá para pedir outra coisa a ela: **quantos gols cada
time vai fazer**. E derivar as três probabilidades disso.

### Por que isso é melhor

Um placar carrega mais informação que um resultado. Para o alvo "vitória do A", um 1×0 e um
5×0 são a mesma coisa — a rede é proibida de aprender a diferença. Prevendo gols, cada
partida ensina o dobro: quanto o ataque de um produz **e** quanto a defesa do outro cede.

Medido nas 5 janelas de validação, com as mesmas 11 entradas:

| Saída da rede | Acurácia | Log loss |
|---|---|---|
| 3 probabilidades (softmax) | 63,4% (±5,3) | 0,826 |
| **2 gols esperados (Poisson)** | **64,9% (±4,0)** | **0,818** |

O ganho é pequeno e está dentro da margem de erro — mas vem acompanhado de duas coisas
concretas: a medição ficou mais estável (±4,0 contra ±5,3) e a rede passa a dizer **placar
provável**, não só quem ganha.

### Como gols viram probabilidades

A rede devolve dois números: os gols esperados de cada time (ex.: 2,1 e 0,9). Gols em futebol
seguem bem de perto uma **distribuição de Poisson** — uma fórmula para eventos raros e
independentes. Com ela dá para calcular a chance de cada placar:

```
P(2×1)  =  P(A fazer 2) × P(B fazer 1)
```

Somando todos os placares em que A fez mais → probabilidade de vitória do A. Todos os
placares iguais → empate. E assim por diante.

### O resto continua igual

| | antes | agora |
|---|---|---|
| Saída | 3 neurônios, `softmax` | 2 neurônios, `exponential` |
| Função de erro | `sparse_categorical_crossentropy` | `poisson` |
| Alvo | 0, 1 ou 2 | os gols de cada time |
| Entradas, camadas, L2, dropout | — | **sem mudança** |

In [ ]:
from scipy.stats import poisson as dist_poisson

MAX_GOLS = 11   # placares de 0x0 até 10x10 cobrem praticamente 100% dos casos


def criar_rede(n_entradas, n1=16, n2=8, l2=0.01, dropout=0.3, lr=0.003):
    """Rede que prevê os GOLS ESPERADOS de cada time (2 saídas, sempre positivas)."""
    keras.backend.clear_session()
    tf.random.set_seed(SEED)
    rede = keras.Sequential([
        layers.Input(shape=(n_entradas,)),
        layers.Dense(n1, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.Dropout(dropout),
        layers.Dense(n2, activation="relu", kernel_regularizer=regularizers.l2(l2)),
        layers.Dropout(dropout),
        layers.Dense(2, activation="exponential"),   # exponential => nunca gera gol negativo
    ])
    rede.compile(optimizer=keras.optimizers.Adam(lr), loss="poisson")
    return rede


def matriz_de_placares(gols_A, gols_B):
    """Chance de cada placar possível. Linha = gols do A, coluna = gols do B."""
    k = np.arange(MAX_GOLS)
    pa = dist_poisson.pmf(k, max(float(gols_A), 1e-6))
    pb = dist_poisson.pmf(k, max(float(gols_B), 1e-6))
    M = np.outer(pa, pb)
    return M / M.sum()


def gols_para_probabilidades(gols):
    """Converte os gols esperados em P(vitória A), P(empate), P(vitória B)."""
    P = np.zeros((len(gols), 3))
    for i, (ga, gb) in enumerate(gols):
        M = matriz_de_placares(ga, gb)
        P[i] = [np.tril(M, -1).sum(),   # A fez mais gols  -> abaixo da diagonal
                np.trace(M),            # placares iguais  -> a diagonal
                np.triu(M, 1).sum()]    # B fez mais gols  -> acima da diagonal
    return P / P.sum(axis=1, keepdims=True)


def treinar_janela(X, gols, ini, fim, cfg, epocas=300, paciencia=25):
    """Treina em [0:ini) e prevê [ini:fim). Últimos 15% do treino viram validação."""
    corte = int(ini * 0.85)
    escala = StandardScaler().fit(X[:corte])          # escala ajustada SÓ no treino
    Xtr, Xva, Xte = (escala.transform(X[:corte]),
                     escala.transform(X[corte:ini]),
                     escala.transform(X[ini:fim]))

    rede = criar_rede(X.shape[1], **cfg)
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=paciencia,
                                           restore_best_weights=True, min_delta=1e-4)
    hist = rede.fit(Xtr, gols[:corte], validation_data=(Xva, gols[corte:ini]),
                    epochs=epocas, batch_size=32, callbacks=[parada], verbose=0)

    gols_previstos = rede.predict(Xte, verbose=0)
    return gols_para_probabilidades(gols_previstos), gols_previstos, rede, escala, hist


def walk_forward(X, y, gols, cfg):
    """Devolve média e oscilação (entre as 5 janelas) da acurácia e do log loss."""
    n = len(y); accs, lls = [], []
    for f in CORTES:
        ini, fim = int(n * f), int(n * (f + 0.10))
        P, *_ = treinar_janela(X, gols, ini, fim, cfg)
        accs.append(accuracy_score(y[ini:fim], P.argmax(1)))
        lls.append(log_loss(y[ini:fim], P, labels=[0, 1, 2]))
    return (float(np.mean(accs)), float(np.std(accs)),
            float(np.mean(lls)),  float(np.std(lls, ddof=1)))


# --- conferência rápida: 2,1 x 0,9 gols deve dar um favorito claro ---
demo = gols_para_probabilidades(np.array([[2.1, 0.9]]))[0]
print("Exemplo: se a rede prevê 2,1 x 0,9 gols, isso vira")
print(f"  vitória A {demo[0]*100:.1f}%  |  empate {demo[1]*100:.1f}%  |  vitória B {demo[2]*100:.1f}%")
M = matriz_de_placares(2.1, 0.9)
i, j = np.unravel_index(M.argmax(), M.shape)
print(f"  placar mais provável: {i} x {j}  ({M[i, j]*100:.1f}% de chance)\n")

criar_rede(11).summary()

## Passo 7 — Busca da melhor configuração

Testa combinações de **janela histórica** × **tamanho da rede**, sempre pela média das
5 janelas walk-forward.

**O critério de escolha tem duas etapas**, e a segunda é o que impede o notebook de repetir
o erro da versão anterior.

1. **Log loss, não acurácia.** Acurácia oscila ±5 pontos por puro acaso; log loss é bem mais
   estável e é o que importa se você quer usar as probabilidades.
2. **Regra de um erro padrão.** Mesmo o log loss empata no topo — as primeiras colocadas
   costumam diferir por 0,002, que é ruído. Escolher a "melhor" nesse caso é escolher a
   sortuda.

   O truque é medir *quanta incerteza a própria medição tem*: o log loss da melhor
   configuração oscila entre as 5 janelas, e essa oscilação dividida por √5 é o **erro
   padrão**. Qualquer configuração dentro dessa margem está estatisticamente empatada com a
   primeira. Entre as empatadas, o notebook fica com a **mais estável** — a que menos oscila
   entre as janelas.

Sem essa segunda etapa, numa execução real a busca escolheu uma configuração com acurácia
64,0% (±5,8) por ter log loss 0,8094, descartando outra com 65,6% (±3,7) e log loss 0,8112.
Uma diferença de 0,0018 não justifica trocar uma configuração visivelmente mais estável.

⏱️ Leva alguns minutos (6 configurações × 5 janelas = 30 treinos).

In [ ]:
GRADE_JANELAS = [3, 5, 8]
GRADE_REDES = [
    {"n1": 16, "n2": 8,  "l2": 0.01, "dropout": 0.3},
    {"n1": 32, "n2": 16, "l2": 0.03, "dropout": 0.4},
]

GOLS = jogos[["GOLS A", "GOLS B"]].values.astype("float32")   # o novo alvo da rede

resultados = []
for janela in GRADE_JANELAS:
    X, y, *_ = montar_features(jogos, janela=janela)
    for cfg in GRADE_REDES:
        acc, desvio, ll, ll_desvio = walk_forward(X, y, GOLS, cfg)
        resultados.append({"janela": janela, "rede": f"{cfg['n1']}-{cfg['n2']}",
                           "l2": cfg["l2"], "dropout": cfg["dropout"],
                           "acuracia": acc, "desvio": desvio,
                           "log_loss": ll, "ll_desvio": ll_desvio,
                           "_cfg": cfg})
        print(f"janela={janela:2d}  rede={cfg['n1']:2d}-{cfg['n2']:2d}  "
              f"acurácia={acc*100:5.1f}% (±{desvio*100:.1f})  log loss={ll:.4f}")

tabela = pd.DataFrame(resultados).sort_values("log_loss").reset_index(drop=True)

# ------------------------------------------------------------------
# ESCOLHA PELA "REGRA DE UM ERRO PADRÃO"
# ------------------------------------------------------------------
# Escolher direto o menor log loss é uma armadilha: as diferenças no topo
# costumam ser de 0,002 — puro ruído. Selecionar por ruído é exatamente o
# erro que derrubou a versão anterior deste projeto.
#
# A regra padrão: junte todas as configurações que estão a menos de um erro
# padrão da melhor — elas são estatisticamente empatadas — e, entre elas,
# fique com a MAIS ESTÁVEL (menor oscilação entre as 5 janelas).

# O erro padrão é a oscilação do log loss da MELHOR configuração ENTRE AS 5 JANELAS,
# dividida por raiz de 5. É a incerteza da própria medição — não a diferença entre linhas.
melhor_ll = tabela.iloc[0]
erro_padrao = melhor_ll["ll_desvio"] / np.sqrt(len(CORTES))
limite = melhor_ll["log_loss"] + erro_padrao
empatadas = tabela[tabela["log_loss"] <= limite]

MELHOR = empatadas.sort_values("desvio").iloc[0]
JANELA, CFG = int(MELHOR["janela"]), MELHOR["_cfg"]

print(f"\nMenor log loss: {melhor_ll['log_loss']:.4f}  |  erro padrão da medição: ±{erro_padrao:.4f}")
print(f"{len(empatadas)} de {len(tabela)} configurações caem dentro dessa margem — "
      f"são empates estatísticos.")
print("Entre elas, fica a mais estável entre as 5 janelas.")

print("\n" + "=" * 58)
print(f"🏆 ESCOLHIDA: janela={JANELA} jogos | rede {MELHOR['rede']} | "
      f"l2={MELHOR['l2']} | dropout={MELHOR['dropout']}")
print(f"   acurácia {MELHOR['acuracia']*100:.1f}% (±{MELHOR['desvio']*100:.1f})  |  "
      f"log loss {MELHOR['log_loss']:.4f}")
print("=" * 58)
print("\n⚠️  Diferenças menores que ~5 pontos de acurácia entre linhas NÃO são conclusivas.")

tabela.drop(columns="_cfg").style.format({"acuracia": "{:.1%}", "desvio": "{:.1%}",
                                          "log_loss": "{:.4f}", "ll_desvio": "{:.4f}"})

## Passo 8 — Termômetro: regressão logística

A mesma validação walk-forward, com as mesmas features, mas com um modelo linear simples.

**Como ler o resultado:**
- Rede **melhor** que a logística → a rede está capturando relações não-lineares reais.
- Rede **empatando** com a logística → é o esperado com ~900 jogos. O gargalo é a
  quantidade de dados, não a arquitetura. Mais camadas não resolvem; mais jogos sim.
- Rede **pior** que a logística → está sobreajustando; aumente `l2` e `dropout`.

In [ ]:
X, y, notas_elo, historico, ultimo_jogo = montar_features(jogos, janela=JANELA)

accs, lls = [], []
for f in CORTES:
    ini, fim = int(N * f), int(N * (f + 0.10))
    esc = StandardScaler().fit(X[:ini])
    lg = LogisticRegression(max_iter=3000).fit(esc.transform(X[:ini]), y[:ini])
    p = lg.predict_proba(esc.transform(X[ini:fim]))
    accs.append(accuracy_score(y[ini:fim], p.argmax(1)))
    lls.append(log_loss(y[ini:fim], p, labels=[0, 1, 2]))

LOG_ACC, LOG_LL = float(np.mean(accs)), float(np.mean(lls))

comparacao = pd.DataFrame([
    {"modelo": "Sempre TIME A (trivial)", "acuracia": BASE_ACC,            "log_loss": np.nan},
    {"modelo": "Média histórica (trivial)", "acuracia": np.nan,            "log_loss": BASE_LL},
    {"modelo": "Regressão logística",     "acuracia": LOG_ACC,             "log_loss": LOG_LL},
    {"modelo": "Rede neural (v3)",        "acuracia": MELHOR["acuracia"],  "log_loss": MELHOR["log_loss"]},
])
print(comparacao.to_string(index=False, na_rep="—",
                           formatters={"acuracia": lambda v: f"{v*100:.1f}%",
                                       "log_loss": lambda v: f"{v:.4f}"}))

if MELHOR["log_loss"] < LOG_LL - 0.01:
    print("\n✅ A rede neural está ganhando da logística.")
elif MELHOR["log_loss"] < LOG_LL + 0.02:
    print("\n➖ Empate técnico com a logística — normal com ~900 jogos.")
    print("   Para a rede se destacar, o caminho é MAIS DADOS (ver plano, seção 5, item 1).")
else:
    print("\n⚠️  A rede está perdendo da logística: aumente l2 e dropout na grade.")

## Passo 9 — Treino final e avaliação honesta

Treina com os primeiros 85% dos jogos e avalia nos **15% finais**, que nunca foram usados
em nenhuma decisão até aqui. Este é o número em que confiar.

Como a rede agora prevê gols, dá para fazer uma checagem que antes não existia: **os gols
previstos batem com os gols reais?** Se a média prevista for 1,6 × 1,1 e a real for 1,7 × 1,1,
a rede entendeu a escala do futebol. Se der 3,5 × 0,2, tem algo errado — mesmo que a acurácia
pareça boa.

In [ ]:
CORTE_FINAL = int(N * 0.85)

prob_teste, gols_teste, rede_final, escala_final, historia = treinar_janela(
    X, GOLS, CORTE_FINAL, N, CFG)
pred_teste = prob_teste.argmax(1)
y_teste = y[CORTE_FINAL:]

acc = accuracy_score(y_teste, pred_teste)
ll = log_loss(y_teste, prob_teste, labels=[0, 1, 2])

print("=" * 58)
print(f"🏁 TESTE FINAL — {len(y_teste)} jogos nunca vistos")
print(f"   Acurácia : {acc*100:.1f}%    (trivial: {(y_teste == 0).mean()*100:.1f}%)")
print(f"   Log loss : {ll:.4f}       (trivial: {BASE_LL:.4f})")
print("=" * 58)

print(f"\n⚽ Gols previstos x gols reais (média nos {len(y_teste)} jogos):")
print(f"   time A: previu {gols_teste[:, 0].mean():.2f}  |  real {GOLS[CORTE_FINAL:, 0].mean():.2f}")
print(f"   time B: previu {gols_teste[:, 1].mean():.2f}  |  real {GOLS[CORTE_FINAL:, 1].mean():.2f}")
print("   (se estes números baterem, a rede entendeu a escala de gols do futebol)")

print("\nDesempenho por tipo de resultado:")
print(classification_report(y_teste, pred_teste,
                            target_names=["Vitória A", "Empate", "Vitória B"],
                            zero_division=0))

mc = confusion_matrix(y_teste, pred_teste, labels=[0, 1, 2])
print("Matriz de confusão (linha = real, coluna = previsto):")
print(pd.DataFrame(mc, index=["real: A", "real: E", "real: B"],
                   columns=["prev: A", "prev: E", "prev: B"]))

p_empate = prob_teste[:, 1]
print(f"\n💡 Sobre o empate: o modelo dá em média {p_empate.mean()*100:.1f}% de chance de empate,")
print(f"   e empates são {(y_teste == 1).mean()*100:.1f}% dos jogos — ou seja, a PROBABILIDADE está certa.")

if (pred_teste == 1).sum() == 0:
    print(f"\n   Mesmo assim ele nunca CRAVA empate (máximo que deu: {p_empate.max()*100:.1f}%).")
    print("   Empate é o que sobra quando dois times são parecidos: não tem perfil próprio,")
    print("   então quase nunca é a maior das três. Trocar a saída para gols não resolveu isso")
    print("   — foi testado. É uma limitação de escolher 'o maior', não do modelo.")
    print("\n   Se você precisa que ele arrisque empates, use uma regra em vez do argmax:")
    print("       palpite = 'Empate' se P(empate) > 0.28 senão o maior dos outros dois")
    print("   Isso sobe o acerto em empates e derruba o acerto geral. É uma troca.")
else:
    print(f"\n   Ele cravou empate em {(pred_teste == 1).sum()} jogos.")

## Passo 10 — Gráficos: treino, comparação e calibração

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(17, 4.5))

# (1) curva de treino
eixos[0].plot(historia.history["loss"], label="treino")
eixos[0].plot(historia.history["val_loss"], label="validação")
eixos[0].set_title("Erro ao longo do treino")
eixos[0].set_xlabel("época"); eixos[0].set_ylabel("erro de Poisson (gols)")
eixos[0].legend(); eixos[0].grid(alpha=.25)

# (2) configurações testadas
eixos[1].scatter(tabela["acuracia"] * 100, tabela["log_loss"], s=70)
for _, r in tabela.iterrows():
    eixos[1].annotate(f"j{int(r['janela'])}·{r['rede']}",
                      (r["acuracia"] * 100, r["log_loss"]), fontsize=8,
                      xytext=(4, 4), textcoords="offset points")
eixos[1].axhline(BASE_LL, ls="--", c="crimson", label="média histórica")
eixos[1].axvline(BASE_ACC * 100, ls="--", c="gray", label="sempre TIME A")
eixos[1].set_xlabel("acurácia (%)"); eixos[1].set_ylabel("log loss  (menor = melhor)")
eixos[1].set_title("Configurações testadas"); eixos[1].legend(fontsize=8); eixos[1].grid(alpha=.25)

# (3) calibração
conf = prob_teste.max(1)
certo = (pred_teste == y_teste).astype(float)
faixas = np.linspace(0.30, 1.00, 8)
xs, ys, ns = [], [], []
for a, b in zip(faixas[:-1], faixas[1:]):
    m = (conf >= a) & (conf < b)
    if m.sum() >= 5:
        xs.append(conf[m].mean()); ys.append(certo[m].mean()); ns.append(int(m.sum()))
eixos[2].plot([0.3, 1], [0.3, 1], "k--", lw=1, label="calibração perfeita")
eixos[2].plot(xs, ys, "o-", label="o modelo")
for x_, y_, n_ in zip(xs, ys, ns):
    eixos[2].annotate(f"n={n_}", (x_, y_), fontsize=8, xytext=(4, -10),
                      textcoords="offset points")
eixos[2].set_xlabel("confiança prevista"); eixos[2].set_ylabel("acerto real")
eixos[2].set_title("Calibração"); eixos[2].legend(fontsize=8); eixos[2].grid(alpha=.25)

plt.tight_layout(); plt.show()

print("Como ler a calibração: se o modelo diz '70% de chance', ele deveria acertar ~70%")
print("das vezes. Pontos ACIMA da linha = modelo modesto demais; ABAIXO = confiante demais.")

## Passo 11 — Prever um jogo novo

Agora a previsão vem em três camadas, da mais informativa para a mais resumida:

1. **Gols esperados** — ex.: `2,1 × 0,9`. É o que a rede realmente calcula.
2. **Placar mais provável** — o quadradinho mais pesado da tabela de placares.
3. **As três probabilidades** — somando os placares de cada tipo.

A função também avisa quando não deve ser levada a sério: quando alguma das seleções tem
histórico curto demais na base.

In [ ]:
def pistas_atuais(time, janela=None):
    q = list(historico.get(time, []))[-(janela or JANELA):]
    if not q:
        return [0.0, 0.0, 0.0, 0.0, 365.0], 0
    dias = (jogos["DATA"].max() - pd.Timestamp(ultimo_jogo[time])).days
    return [sum(x[0] for x in q) / len(q),
            sum(x[1] for x in q) / len(q),
            sum(x[2] for x in q) / len(q),
            float(len(q)),
            float(min(dias, 365))], len(q)


def prever(time_A, time_B, mostrar_placares=True):
    A, B = normalizar(time_A), normalizar(time_B)

    faltando = [t for t in (A, B) if t not in notas_elo]
    if faltando:
        print(f"⚠️  Não encontrei na base: {', '.join(faltando)}")
        parecidos = [s for s in notas_elo if any(f[:4] in s for f in faltando)][:6]
        if parecidos:
            print("   Você quis dizer:", ", ".join(parecidos), "?")
        return

    fa, na = pistas_atuais(A)
    fb, nb = pistas_atuais(B)
    entrada = np.array([[notas_elo[A] - notas_elo[B]] + fa + fb], dtype=np.float32)

    gols = rede_final.predict(escala_final.transform(entrada), verbose=0)[0]
    p = gols_para_probabilidades(gols.reshape(1, 2))[0]
    M = matriz_de_placares(gols[0], gols[1])

    print(f"\n  {A}  ×  {B}")
    print(f"  Elo: {notas_elo[A]:.0f} × {notas_elo[B]:.0f}"
          f"   |   histórico: {na} × {nb} jogos")

    print(f"\n  ⚽ Gols esperados:   {gols[0]:.1f}  ×  {gols[1]:.1f}")
    i, j = np.unravel_index(M.argmax(), M.shape)
    print(f"  🎯 Placar mais provável:  {i} × {j}   ({M[i, j]*100:.1f}% de chance)\n")

    for rotulo, prob in zip([f"Vitória {A}", "Empate", f"Vitória {B}"], p):
        print(f"  {rotulo:<26} {prob*100:5.1f}%  {'█' * int(round(prob * 30))}")

    if mostrar_placares:
        chances = [(M[a, bb] * 100, a, bb) for a in range(5) for bb in range(5)]
        chances.sort(reverse=True)
        print("\n  Placares mais prováveis: " +
              "   ".join(f"{a}×{bb} ({c:.0f}%)" for c, a, bb in chances[:5]))

    total_jogos = pd.concat([jogos['TIME A'], jogos['TIME B']]).value_counts()
    fracos = [t for t in (A, B) if total_jogos.get(t, 0) < 5]
    if fracos:
        print(f"\n  ⚠️  {' e '.join(fracos)} tem menos de 5 jogos na base — "
              f"esta previsão é pouco confiável.")
    if p.max() < 0.45:
        print("  ℹ️  Jogo equilibrado: o modelo não tem favorito claro.")


prever("BRASIL", "MARROCOS")
prever("ESPANHA", "ARGENTINA")


## Passo 12 — Salvar o modelo e a base limpa

Guarda o modelo treinado e a base consolidada, para não ter que refazer tudo depois.

In [ ]:
import os

PASTA_MODELOS = "04_modelos" if os.path.isdir("04_modelos") else "."
caminho_modelo = os.path.join(PASTA_MODELOS, "modelo_prever_jogos_v3.keras")
rede_final.save(caminho_modelo)

with pd.ExcelWriter("base_jogos_limpa.xlsx", engine="openpyxl") as w:
    (jogos.assign(RESULTADO=jogos["ALVO"].map({0: "A", 1: "E", 2: "B"}))
          [["DATA", "TIME A", "TIME B", "GOLS A", "GOLS B", "RESULTADO"]]
          .to_excel(w, sheet_name="jogos", index=False))

    ranking = (pd.Series(notas_elo).sort_values(ascending=False).round(0)
                 .rename("ELO").rename_axis("SELECAO").reset_index())
    ranking.insert(0, "POSICAO", range(1, len(ranking) + 1))
    contagem = pd.concat([jogos["TIME A"], jogos["TIME B"]]).value_counts()
    ranking["JOGOS_NA_BASE"] = ranking["SELECAO"].map(contagem).astype(int)
    ranking.to_excel(w, sheet_name="ranking_elo_do_modelo", index=False)

print("Salvos:")
print("  •", caminho_modelo)
print("  • base_jogos_limpa.xlsx  (abas: jogos + ranking_elo_do_modelo)")

print("\n🏆 Top 15 do ranking Elo — só seleções com 10+ jogos na base:")
print(ranking[ranking["JOGOS_NA_BASE"] >= 10].head(15).to_string(index=False))

try:
    from google.colab import files
    files.download(caminho_modelo)
    files.download("base_jogos_limpa.xlsx")
except ImportError:
    pass

---

## Próximos passos (em ordem de retorno)

1. **Mais jogos.** 906 partidas é o principal limitador. Bases públicas de resultados
   internacionais têm dezenas de milhares de jogos. Com ~20.000, a rede neural passa a ter
   espaço real para superar a regressão logística.
2. **Coluna de mando de campo** (`Casa` / `Fora` / `Neutro`). O fator casa é um dos
   preditores mais fortes do futebol e hoje o projeto simplesmente não o tem — a coluna
   `TIME A` às vezes é o mandante, às vezes não.
3. **Coluna de tipo de competição** (amistoso / eliminatória / copa). Times jogam de forma
   diferente em amistosos; misturar tudo adiciona ruído.
4. **Modelo de gols (Poisson)** em vez de classificação direta — costuma calibrar melhor
   os empates.
5. **Embeddings de seleção** — só faz sentido depois do item 1.

**Expectativa realista:** casas de apostas profissionais acertam 50–55% em resultado 1×2.
Nenhum modelo chega a 80–90%. Se algum resultado indicar isso, é vazamento de dados.